### 1.1.11. Subsets and Slices of Ordered Pairs

In many practical optimization models the relevant index set is not the full Cartesian product
of two sets but only a **sparse subset** of pairs — for example, the origin–destination
combinations for which a shipping route actually exists.  AMPL addresses this in two
complementary ways:

1. **Subset declarations** — declare a 2-D set as a subset (`within`) of a Cartesian product,
   or derive a filtered subset using the `setof` operator.
2. **Slice notation** — when one component of a tuple is already fixed (by an enclosing
   indexing expression or a literal value), AMPL automatically interprets `{(i,j) in LINKS}`
   as the set of all values of the free component, effectively "slicing" through the 2-D set
   along the fixed component.

Together these two mechanisms are the standard AMPL technique for handling sparse
multi-dimensional index sets efficiently and correctly.

In [ ]:
# ── Formal syntax ──────────────────────────────────────────────────────────────

# 1.  Subset declaration with 'within'
#     set NAME within { SET1, SET2 };
#     set NAME within SET1 cross SET2;

# 2.  Filtered subset using setof
#     set NAME = setof { dummy-members in PARENT-SET : condition } (member-expr) ;

# 3.  Slice notation inside an indexing expression
#     When component i is already defined:
#         { (i, j) in LINKS }   -- iterates over all j such that (i,j) in LINKS
#         { (i, j) in LINKS }   -- iterates over all i such that (i,j) in LINKS
#                                   (when j is the already-defined component)

# 4.  Literal slice — fix one component to a specific value
#         { ("GARY", j) in LINKS }  -- all j reachable from GARY
#         { (i, "DET")  in LINKS }  -- all i that ship to DET

# 5.  Derived indexed-set collection from a 2-D set
#     set DEST_OF {i in ORIG} = setof { (i, j) in LINKS } j;

#### Conceptual explanation

**Subset declarations.**  When a 2-D set is declared with

```ampl
set LINKS within {ORIG, DEST};
```

AMPL requires that every pair supplied in the data file (or via amplpy) satisfies
`(i,j) in ORIG cross DEST`.  The `within` attribute is a *membership constraint*: it does
not determine the set's members (those come from data), but it validates them at load time
and makes the model's intent explicit.

The equivalent form using the `cross` operator,

```ampl
set LINKS within ORIG cross DEST;
```

is semantically identical.  Both forms are accepted by AMPL; the brace form
`within {ORIG, DEST}` is the one most commonly seen in AMPL literature.

**Filtered subsets via `setof`.**  The `setof` operator constructs a new set by evaluating
an expression for each member of an indexing set and collecting all distinct results:

```ampl
setof { dummy-members in PARENT : condition } member-expr
```

Duplicate values are silently discarded, so the result is always a proper set.  A `setof`
expression may appear anywhere a set expression is valid: in a `=` definition, as the
right-hand side of a `within`, in a `display` command, or in an indexing expression.

**Slice notation.**  Slice notation is not a separate keyword — it is the consequence of
how AMPL resolves the indexing expression `{(i,j) in LINKS}` when one of the dummy
indices is already bound in the surrounding scope.

Concretely, inside

```ampl
subject to Supply {i in ORIG}:
    sum {(i,j) in LINKS} Trans[i,j] = supply[i];
```

the outer `{i in ORIG}` binds `i` to a specific origin.  Inside that constraint body,
the expression `{(i,j) in LINKS}` therefore denotes the **set of all `j` such that the
pair `(i,j)` is a member of `LINKS`** — a one-dimensional slice of the 2-D set in the
row corresponding to `i`.  AMPL uses an efficient internal index on `LINKS` to locate
these pairs; the modeler does not need to build a separate data structure.

The same mechanism applies when the second component is already defined:

```ampl
subject to Demand {j in DEST}:
    sum {(i,j) in LINKS} Trans[i,j] = demand[j];
```

Here `j` is fixed and `{(i,j) in LINKS}` iterates over all origins `i` such that
`(i,j) in LINKS` — a column slice.

The fixed component may also be a **literal** rather than a dummy index:

```ampl
sum {("GARY", j) in LINKS} Trans["GARY", j]
```

This is identical in semantics to fixing `i = "GARY"` in the row-slice example above.

#### Attribute and operator reference

| Construct | Syntax | Result | Notes |
|---|---|---|---|
| Subset declaration | `set S within {A, B};` | declares `S` as a 2-D set; data must lie within `A cross B` | validates data at load time |
| Subset declaration (cross form) | `set S within A cross B;` | same as above | both forms accepted |
| Membership test | `(i,j) in LINKS` | Boolean: 1 if the pair is a member | usable in conditions, `check`, `if` |
| `setof` | `setof {(i,j) in S: cond} (i,j)` | new set of all tuples satisfying `cond` | dimension = number of components in member-expr |
| `setof` (projection) | `setof {(i,j,p) in R} (i,j)` | projects 3-D set to 2-D by discarding `p` | duplicates silently removed |
| Row slice | `{(i,j) in LINKS}` with `i` pre-bound | set of `j` values paired with the fixed `i` | AMPL uses internal index for efficiency |
| Column slice | `{(i,j) in LINKS}` with `j` pre-bound | set of `i` values paired with the fixed `j` | same mechanism, different fixed component |
| Literal slice | `{("GARY",j) in LINKS}` | set of `j` values for which `("GARY",j) in LINKS` | fixed component is a constant expression |
| Membership test on a literal pair | `("GARY","DET") in LINKS` | Boolean: 1 or 0 | not a slice; returns a scalar |
| Derived indexed-set collection | `set DEST_OF {i in ORIG} = setof {(i,j) in LINKS} j;` | `DEST_OF[i]` = set of destinations reachable from `i` | indexed family of 1-D sets |

#### Example 1 — Sparse transportation model with `within` and slice indexing

This is the canonical AMPL example from Fourer, Gay, and Kernighan (Figure 6-2a).
Shipments are possible only on a sparse set of origin–destination links.  The `within`
attribute validates data, and slice notation keeps the constraint declarations concise.

In [ ]:
# ── transp3.mod ────────────────────────────────────────────────────────────────
set ORIG;                              # origin cities
set DEST;                              # destination cities
set LINKS within {ORIG, DEST};         # sparse subset of (orig, dest) pairs

param supply {ORIG} >= 0;              # supply available at each origin
param demand {DEST} >= 0;              # demand required at each destination

check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];

param cost  {LINKS} >= 0;              # shipment cost per unit — defined on LINKS only
var   Trans {LINKS} >= 0;              # units shipped — defined on LINKS only

minimize Total_Cost:
    sum {(i,j) in LINKS} cost[i,j] * Trans[i,j];

# Here {(i,j) in LINKS} with i pre-bound: row slice — iterates over j reachable from i
subject to Supply {i in ORIG}:
    sum {(i,j) in LINKS} Trans[i,j] = supply[i];

# Here {(i,j) in LINKS} with j pre-bound: column slice — iterates over i that ship to j
subject to Demand {j in DEST}:
    sum {(i,j) in LINKS} Trans[i,j] = demand[j];

In [ ]:
# ── transp3.dat ────────────────────────────────────────────────────────────────
data;

param: ORIG: supply :=
    GARY  1400
    CLEV  2600
    PITT  2900 ;

param: DEST: demand :=
    FRA   900
    DET  1200
    LAN   600
    WIN   400
    STL  1700
    FRE  1100
    LAF  1000 ;

# Pairs listed here automatically define the membership of LINKS
param: LINKS: cost :=
    GARY DET  14    GARY LAN  11    GARY STL  16    GARY LAF   8
    CLEV FRA  27    CLEV DET   9    CLEV LAN  12    CLEV WIN   9
    CLEV STL  26    CLEV LAF  17
    PITT FRA  24    PITT WIN  13    PITT STL  28    PITT FRE  99 ;

In this model the indexing expression `{(i,j) in LINKS}` appears in three places and
means something different each time:

| Location | What is pre-bound | What the expression iterates over |
|---|---|---|
| `sum {(i,j) in LINKS}` in the objective | nothing — both `i` and `j` are free | all pairs in `LINKS` |
| `sum {(i,j) in LINKS}` in `Supply` | `i` (fixed by the outer `{i in ORIG}`) | all destinations `j` reachable from `i` |
| `sum {(i,j) in LINKS}` in `Demand` | `j` (fixed by the outer `{j in DEST}`) | all origins `i` that ship to `j` |

The constraint written without slice notation,

```ampl
subject to Supply {i in ORIG}:
    sum {j in DEST} Trans[i,j] = supply[i];   # WRONG: Trans[i,j] undefined for j not in LINKS
```

would cause AMPL to attempt to evaluate `Trans['GARY','FRA']`, which does not exist,
producing an `invalid subscript` error at model-generation time.

#### Example 2 — `setof` to build filtered subsets

The `setof` operator constructs a set by evaluating a member expression for each element
of an indexing set (optionally filtered by a condition).  The three patterns below show
its most common uses with 2-D sets.

In [ ]:
set ORIG;                                   # origins
set DEST;                                   # destinations
set LINKS within {ORIG, DEST};              # all allowed pairs
param cost {LINKS} >= 0;                    # shipping cost per unit
param threshold >= 0;                       # cost cutoff

# Pattern 1: filtered subset of pairs — keep only cheap links
set CHEAP_LINKS = setof {(i,j) in LINKS: cost[i,j] < threshold} (i,j);

# Pattern 2: projection — extract the set of all origins that have at least one link
#   (pairs with p != q removed; result is 1-D)
set ACTIVE_ORIG = setof {(i,j) in LINKS} i;
set ACTIVE_DEST = setof {(i,j) in LINKS} j;

# Pattern 3: extract pairs from a 3-D set by ignoring the third component
set ROUTES dimen 3;                         # triples (orig, dest, product)
set LINKS_3D = setof {(i,j,p) in ROUTES} (i,j);  # 2-D set: every (i,j) used by any product
set PROD_3D  = setof {(i,j,p) in ROUTES} p;      # 1-D set: every product that appears

# Pattern 4: derive a set using a condition on a different parameter
#   DEMAND contains (dest, product) pairs with positive demand
set DEST2;
set PROD2;
param demand2 {DEST2, PROD2} >= 0;
set DEMAND = setof {j in DEST2, p in PROD2: demand2[j,p] > 0} (j,p);

#### Example 3 — Derived indexed-set collection and network flow constraints

A common idiom in network models is to define, for each origin `i`, the set of destinations
reachable from it.  This can be expressed either as an indexed family of 1-D sets (derived
from `LINKS` using `setof`) or implicitly through slice notation in the constraints.

Both representations are shown below; they produce identical mathematical models.

In [ ]:
# ── network_flow.mod ───────────────────────────────────────────────────────────
set ORIG;                                   # source nodes
set DEST;                                   # sink nodes
set LINKS within {ORIG, DEST};              # allowed arcs

param supply {ORIG} >= 0;
param demand {DEST} >= 0;
param cost   {LINKS} >= 0;

var Trans {LINKS} >= 0;

# ── Representation A: explicit indexed-set collection ──────────────────────────
#
# DEST_OF[i] = set of all destinations reachable from origin i
# ORIG_OF[j] = set of all origins that can ship to destination j

set DEST_OF {i in ORIG} = setof {(i,j) in LINKS} j;
set ORIG_OF {j in DEST} = setof {(i,j) in LINKS} i;

subject to Supply_A {i in ORIG}:
    sum {j in DEST_OF[i]} Trans[i,j] = supply[i];

subject to Demand_A {j in DEST}:
    sum {i in ORIG_OF[j]} Trans[i,j] = demand[j];

# ── Representation B: slice notation (equivalent; more compact) ─────────────────

subject to Supply_B {i in ORIG}:
    sum {(i,j) in LINKS} Trans[i,j] = supply[i];   # same as sum over DEST_OF[i]

subject to Demand_B {j in DEST}:
    sum {(i,j) in LINKS} Trans[i,j] = demand[j];   # same as sum over ORIG_OF[j]

minimize Total_Cost:
    sum {(i,j) in LINKS} cost[i,j] * Trans[i,j];

Representations A and B produce exactly the same model.  Representation B (slice notation)
is preferred in practice because:

- It requires no additional set declarations.
- AMPL uses an efficient internal inverted index on `LINKS` to find the relevant pairs;
  for large sparse sets this is substantially faster than explicitly checking every
  `j in DEST`.

Representation A with explicit `DEST_OF` is useful when the derived sets are needed in
multiple places or when the Python/amplpy layer needs to inspect which destinations are
reachable from a given origin.

#### Example 4 — Literal slice and membership test

A literal (constant) value may replace either dummy index in a slice expression.  A pair
with **both** components fixed is a membership test that returns a Boolean, not a set.

In [ ]:
set ORIG;
set DEST;
set LINKS within {ORIG, DEST};
param cost  {LINKS} >= 0;
var   Trans {LINKS} >= 0;

# Literal row slice: set of all j such that ("GARY",j) in LINKS
display setof {("GARY",j) in LINKS} j;    # {DET, LAN, STL, LAF} given transp3 data

# Equivalent inline sum over the literal slice:
display sum {("GARY",j) in LINKS} cost["GARY",j];  # total cost out of GARY

# Literal column slice: set of all i such that (i,"DET") in LINKS
display setof {(i,"DET") in LINKS} i;     # {GARY, CLEV} given transp3 data

# Boolean membership test — BOTH components fixed; returns 1 or 0, NOT a set
display ("GARY","DET")  in LINKS;         # 1  (True)
display ("GARY","FRA")  in LINKS;         # 0  (False)

# Membership test inside a condition:
param is_direct {i in ORIG, j in DEST} = if (i,j) in LINKS then 1 else 0;

#### Example 5 — Slices in 3-D sets (multicommodity transportation)

Slice notation extends naturally to sets of triples and longer tuples.  In the
multicommodity transportation model, the set `ROUTES` contains triples
`(origin, dest, product)`.  Three different slice patterns are required for the
three constraint families.

In [ ]:
# ── multi_transp.mod ───────────────────────────────────────────────────────────
set ORIG;
set DEST;
set PROD;
set ROUTES within {ORIG, DEST, PROD};   # sparse (origin, dest, product) triples

param supply {ORIG, PROD} >= 0;
param demand {DEST, PROD} >= 0;
param limit  {ORIG, DEST} >= 0;        # capacity on each (orig,dest) arc
param cost   {ROUTES} >= 0;

var Trans {ROUTES} >= 0;

minimize Total_Cost:
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

# Slice: i and p pre-bound — iterates over all j such that (i,j,p) in ROUTES
subject to Supply {i in ORIG, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];

# Slice: j and p pre-bound — iterates over all i such that (i,j,p) in ROUTES
subject to Demand {j in DEST, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = demand[j,p];

# Slice: i and j pre-bound — iterates over all p such that (i,j,p) in ROUTES
subject to Multi {i in ORIG, j in DEST}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] <= limit[i,j];

The alternative form for the `Supply` constraint without slice notation is

```ampl
subject to Supply {i in ORIG, p in PROD}:
    sum {j in DEST: (i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];
```

Both forms are correct.  The slice version (`{(i,j,p) in ROUTES}`) is preferred because
AMPL can retrieve the relevant triples directly from an internal index on `ROUTES`, rather
than iterating over all `j in DEST` and checking membership one by one.  For realistically
sized problems the slice form may be the only one that can be processed in reasonable
time and memory (Fourer, Gay, Kernighan §6.3).

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Always declare sparse 2-D sets with <code>within {ORIG, DEST}</code> rather than
<code>dimen 2</code> alone.  The <code>within</code> attribute causes AMPL to validate
every data-supplied pair against the component sets immediately at data-load time,
producing an informative error message rather than a silent wrong answer when data
contains a typographical error such as an unrecognised city name.
</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use slice notation <code>{(i,j) in LINKS}</code> in sums and constraint indexing rather
than the explicit membership-test form <code>{j in DEST: (i,j) in LINKS}</code>.  The
slice form allows AMPL to apply an efficient internal lookup on the 2-D set, which
becomes critical for large sparse models.  The explicit form iterates over the entire
<code>DEST</code> set and tests membership for every element.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Do not write a sum over the full Cartesian product when the variable or parameter is
defined only on the sparse subset.  The constraint

<pre><code>subject to Supply {i in ORIG}:
    sum {j in DEST} Trans[i,j] = supply[i];   # WRONG
</code></pre>

causes AMPL to attempt to evaluate <code>Trans[i,j]</code> for every <code>j in DEST</code>,
including pairs not in <code>LINKS</code>.  AMPL will report <em>invalid subscript</em>
at model-generation time.  The correct form is <code>sum {(i,j) in LINKS}</code>.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Fixing both components of a pair produces a Boolean membership test, not a set.
The expression <code>("GARY", "DET") in LINKS</code> evaluates to <code>1</code> or
<code>0</code> and cannot be used as an index set.  Only one component should be fixed
in a slice expression; the other must remain a free dummy index for AMPL to iterate over.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
The <code>setof</code> operator silently discards duplicate values.  When projecting a
3-D set onto its first two components with
<code>setof {(i,j,p) in ROUTES} (i,j)</code>,
the result is the set of distinct <code>(i,j)</code> pairs, not a multiset.  This is the
intended behaviour, but it means <code>card(setof ...)</code> may be smaller than the
number of iterations of the <code>setof</code> expression.
</div>

#### amplpy equivalents

The table below maps the most common AMPL 2-D set and slice operations to their amplpy
counterparts.  Note that slice notation is an AMPL concept applied inside AMPL models;
there is no direct amplpy method for it.  However, similar filtering operations on 2-D
sets can be performed in Python by working with the list of tuples returned by `.members()`.

| AMPL operation | amplpy equivalent |
|---|---|
| `set LINKS = list of (i,j) pairs;` via data file | `ampl.set["LINKS"] = list_of_tuples` |
| `set LINKS = list of (i,j) pairs;` from dict keys | `ampl.set["LINKS"] = list(cost_dict.keys())` |
| Read all pairs in `LINKS` | `ampl.set["LINKS"].members()` — returns list of tuples |
| Cardinality `card(LINKS)` | `ampl.set["LINKS"].size()` |
| Membership test `("GARY","DET") in LINKS` | `ampl.get_value('("GARY","DET") in LINKS')` returns 1.0 or 0.0 |
| Row slice `{("GARY",j) in LINKS}` as Python set | `{j for (i,j) in ampl.set["LINKS"].members() if i == "GARY"}` |
| Derived set `DEST_OF["GARY"]` | `ampl.get_set("DEST_OF")["GARY"].members()` |
| Assign 2-D set from DataFrame | `ampl.set["LINKS"] = df[["orig","dest"]].apply(tuple, axis=1).tolist()` |
| Read 2-D set as DataFrame | `ampl.set["LINKS"].get_values().to_pandas()` |

In [ ]:
# amplpy: populate LINKS from a cost dictionary and work with the result

from amplpy import AMPL

ampl = AMPL()

ampl.eval("""
    set ORIG;
    set DEST;
    set LINKS within {ORIG, DEST};
    param cost {LINKS} >= 0;
    var   Trans {LINKS} >= 0;

    set DEST_OF {i in ORIG} = setof {(i,j) in LINKS} j;
    set ORIG_OF {j in DEST} = setof {(i,j) in LINKS} i;
""")

ORIG = ["GARY", "CLEV", "PITT"]
DEST = ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"]
cost_dict = {
    ("GARY", "DET"): 14, ("GARY", "LAN"): 11, ("GARY", "STL"): 16, ("GARY", "LAF"): 8,
    ("CLEV", "FRA"): 27, ("CLEV", "DET"): 9,  ("CLEV", "LAN"): 12, ("CLEV", "WIN"): 9,
    ("CLEV", "STL"): 26, ("CLEV", "LAF"): 17,
    ("PITT", "FRA"): 24, ("PITT", "WIN"): 13, ("PITT", "STL"): 28, ("PITT", "FRE"): 99,
}

ampl.set["ORIG"]  = ORIG
ampl.set["DEST"]  = DEST
ampl.set["LINKS"] = list(cost_dict.keys())   # list of (origin, destination) tuples
ampl.param["cost"] = cost_dict

# Read back all pairs in LINKS
all_links = ampl.set["LINKS"].members()      # list of (str, str) tuples
print("Number of links:", ampl.set["LINKS"].size())  # 14

# Retrieve the DEST_OF indexed set for GARY
gary_dests = ampl.get_set("DEST_OF")["GARY"].members()
print("GARY can ship to:", gary_dests)        # ['DET', 'LAN', 'STL', 'LAF']

# Python-side row slice equivalent (no AMPL call needed)
gary_dests_py = {j for (i, j) in cost_dict.keys() if i == "GARY"}
print("Same result (Python):", gary_dests_py)

# Membership test via AMPL expression evaluation
in_links = ampl.get_value('("GARY","DET") in LINKS')   # returns 1.0
not_in   = ampl.get_value('("GARY","FRA") in LINKS')   # returns 0.0
print("GARY->DET in LINKS:", bool(in_links))   # True
print("GARY->FRA in LINKS:", bool(not_in))     # False

#### Edge Cases

**Edge case 1: Origin present in `ORIG` but absent from all links.**

When an origin `i` exists in `ORIG` but has no pairs in `LINKS`, the slice
`{(i,j) in LINKS}` is empty.  A `sum` over an empty set evaluates to zero; a constraint
indexed over an empty set generates zero instances (not an error).  The supply constraint
for that origin will therefore read `0 = supply[i]`, which is satisfied only when
`supply[i] = 0`.  A `check` statement can guard against silent infeasibility caused by
isolated nodes with positive supply.

In [ ]:
set ORIG;
set DEST;
set LINKS within {ORIG, DEST};
param supply {ORIG} >= 0;

# Guard: every origin with positive supply must appear in at least one link
check {i in ORIG: supply[i] > 0}: card(setof {(i,j) in LINKS} j) > 0;

# The Supply constraint for an isolated origin (supply = 0) generates 0 = 0, which
# is trivially satisfied.  For supply > 0 the check above will catch the error first.
var Trans {LINKS} >= 0;
subject to Supply {i in ORIG}:
    sum {(i,j) in LINKS} Trans[i,j] = supply[i];

**Edge case 2: `setof` over an empty parent set.**

When the parent set iterated by `setof` is empty, the result is the empty set `{}`.  No
error is raised.  This is consistent with the standard mathematical definition: `setof`
over an empty domain yields an empty collection.

In [ ]:
set ORIG;
set DEST;
set LINKS within {ORIG, DEST};     # LINKS will be empty for this instance

# setof over an empty LINKS produces the empty set — no error
set ACTIVE_ORIG = setof {(i,j) in LINKS} i;   # {}
set ACTIVE_DEST = setof {(i,j) in LINKS} j;   # {}

display card(ACTIVE_ORIG);   # 0
display card(ACTIVE_DEST);   # 0

# A constraint indexed over ACTIVE_ORIG generates zero instances — not an error
param cost {LINKS} >= 0;
var   Trans {LINKS} >= 0;
subject to NoCost {i in ACTIVE_ORIG}:    # zero constraints; valid model
    sum {(i,j) in LINKS} Trans[i,j] = 0;

# Data for this edge case:
# data;
# set ORIG := GARY CLEV PITT ;
# set DEST := FRA DET LAN ;
# set LINKS :=  ;   # empty — no routes defined

**Edge case 3: `within` constraint violated by data.**

If the data file contains a pair that is not in `ORIG cross DEST`, AMPL raises an error
at data-load time — not at solve time.  The `within` attribute is therefore a lightweight
data validation mechanism.  Without it (using only `dimen 2`), an invalid pair would
silently enter the set and might cause an `invalid subscript` error only later during
constraint generation, making the mistake harder to diagnose.

In [ ]:
set ORIG;                              # {GARY, CLEV, PITT}
set DEST;                              # {FRA, DET, LAN}

# Form 1: with 'within' — AMPL validates every pair at data-load time
set LINKS_SAFE  within {ORIG, DEST};

# Form 2: without 'within' — no validation; bad pairs silently enter the set
set LINKS_UNSAFE dimen 2;

# If the data contains ("GARY", "CHICAGO") and "CHICAGO" is not in DEST:
#   LINKS_SAFE  -> error at data load: "CHICAGO is not in DEST"
#   LINKS_UNSAFE -> no error at data load; error (or wrong result) only later

# Data that would trigger the error:
# data;
# set ORIG := GARY CLEV PITT ;
# set DEST := FRA DET LAN ;
# param: LINKS_SAFE: cost :=   # this line would cause an immediate error:
#     GARY DET 14
#     GARY CHICAGO 999 ;       # CHICAGO not in DEST

**Edge case 4: `setof` interaction with `within` — defining a subset of a subset.**

A set built by `setof` with a condition is a subset of the set being iterated.  It can be
further constrained with a `within` clause to document that fact and catch any future
regression if the condition is inadvertently loosened.

In [ ]:
set ORIG;
set DEST;
set LINKS  within {ORIG, DEST};
param cost {LINKS} >= 0;
param threshold >= 0;

# CHEAP_LINKS is a subset of LINKS (which is itself a subset of ORIG cross DEST)
# The 'within LINKS' clause documents the containment and validates it at model-build time.
set CHEAP_LINKS within LINKS
    = setof {(i,j) in LINKS: cost[i,j] < threshold} (i,j);

# Because CHEAP_LINKS is defined by '=' it cannot receive data from a data file.
# It is recomputed automatically whenever cost or threshold change.

# Use CHEAP_LINKS to restrict a variable or constraint to low-cost routes only:
var   Cheap_Trans {CHEAP_LINKS} >= 0;
subject to BudgetRoute {(i,j) in CHEAP_LINKS}:
    Cheap_Trans[i,j] <= 500;

#### Cross-references

- See also: [1.1.4. Set operations](<1_1_4 set_operations.ipynb>) — `union`, `inter`,
  `diff`, `cross`, and `within` on general sets.
- See also: [1.1.5. Unordered sets](<1_1_5 unordered_sets.ipynb>) — ordering attributes
  and why 2-D sets cannot be declared `ordered`.
- See also: [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>)
  — the `=` vs. `:=` vs. `default` distinction relevant to `setof`-defined sets.
- Source: Fourer, Gay, Kernighan, *AMPL: A Modeling Language for Mathematical Programming*,
  §6.2 (Subsets and Slices of Ordered Pairs) and §6.3 (Sets of Longer Tuples).